In [2]:
ROOT_PATH = 'C:/Users/khoan/OneDrive/Documents/stock_data_scraper'
import os
os.chdir(ROOT_PATH)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
from utils.generic_utils import SQLModule
import pandas as pd
import spacy
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Australia

In [9]:
engine = SQLModule.get_engine(country = 'australia')

## By industry

In [103]:
# Collect from 'company_section.ipynb'
SECTOR = 'Technology'
INDUSTRY = 'Software—Application'

In [104]:
# Get distribution of metrics of company within the industry
query = f"""
    SELECT
        company_name,
        stock_code,
        fundamental->>'longBusinessSummary' AS summary
    FROM stock_info
    WHERE
        fundamental->>'sector' = '{SECTOR}'
        AND
        fundamental->>'industry' = '{INDUSTRY}'
"""
df = pd.read_sql_query(query, engine).set_index('stock_code')
df

,company_name,summary
stock_code,,
AR9,Archtis Ltd,archTIS Limited engages in the design and deve...
BGE,Bridge SaaS Ltd.,Bridge SaaS Limited provides Software-as-a-Ser...
BTH,Bigtincan Holdings Ltd,Bigtincan Holdings Limited develops software f...
C79,Chrysos Corp Ltd,Chrysos Corporation Limited engages in the dev...
CF1,Complii Fintech Solutions Ltd,Complii FinTech Solutions Ltd operates integra...
...,...,...
RUL,RPMGlobal Holdings Ltd,RPMGlobal Holdings Limited develops and provid...
SDR,SiteMinder Limited,"SiteMinder Limited develops, markets, and sell..."
SIS,Simble Solutions Limited,Simble Solutions Limited develops and provides...


In [105]:
nlp = spacy.load('en_core_web_md')
df['encoded'] = df['summary'].apply(lambda x : nlp(x) if not pd.isna(x) else None)
encoded_stock = df.loc['TNE', 'encoded']
df['cosine_sim'] = df['encoded'].apply(lambda x : x.similarity(encoded_stock) if not pd.isna(x) else None)
business_codes = df[(df['cosine_sim'] >= 0.95)].index.tolist()

In [31]:
# Get distribution of metrics of company within the industry
query = f"""
    SELECT
        company_name,
        stock_code,
        fundamental,
        fundamental->>'industry' AS industry,
        fundamental->>'marketCap' AS market_cap,
        fundamental->>'priceToBook' AS pb_ratio,
        fundamental->>'profitMargins' AS profit_margins,
        fundamental->>'trailingEps' AS earning_per_share,
        fundamental->>'enterpriseToEbitda' AS evebitda,
        fundamental->>'trailingPegRatio' AS peg_ratio,
        fundamental->>'debtToEquity' AS de_ratio
    FROM stock_info
    WHERE
        stock_code IN {tuple(business_codes)}

"""
df = pd.read_sql_query(query, engine)
df.head(5)

,company_name,stock_code,fundamental,industry,market_cap,pb_ratio,profit_margins,earning_per_share,evebitda,peg_ratio,de_ratio
0,Algorae Pharmaceuticals Limited,1AI,"{'address1': 'Rialto South Tower', 'address2':...",Biotechnology,10124340,3.0,None,None,-5.511,None,None
1,Auctus Alternative Investments Ltd,AVC,"{'address1': '101 Collins Street', 'address2':...",Asset Management,45358372,1.6966966,0.26143,0.03,22.253,None,5.903
2,Artrya Ltd,AYA,"{'address1': '1257 Hay Street', 'city': 'West ...",Health Information Services,45016880,2.2663553,None,-0.18,-2.253,None,5.647
3,Beamtree Holdings Ltd,BMT,"{'address1': '5 Blue Street', 'address2': 'Sui...",Health Information Services,81133080,1.7177914,-0.18499,-0.02,-15.117,None,3.522
4,Brainchip Holdings Ltd,BRN,"{'address1': '210 George Street', 'address2': ...",Semiconductors,653001280,39.374996,None,-0.02,-20.609,None,9.417


In [110]:
# Rerun the histogram, but add the company in it
fig = make_subplots(rows = 4, cols = 2)
metrics = ['market_cap', 'pb_ratio', 'profit_margins', 'earning_per_share', 'evebitda', 'peg_ratio', 'de_ratio']

for i,metric in enumerate(metrics):
    fig.add_trace(
        go.Histogram(
            x = df[metric].apply(lambda x : float(x) if not pd.isna(x) else None), 
            marker = dict(color = 'blue'),
            showlegend = False
        ),
        row = (i // 2) + 1, col = (i % 2) + 1
    )
    fig.update_xaxes(title = metric, row = (i // 2) + 1, col = (i % 2) + 1)
fig.update_yaxes(showgrid = True, gridcolor = 'gray', title = 'Number of company')

fig.update_layout(
    width = 1000,
    height = 1000,
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    font = dict(family = 'Lato', size = 18)
)
fig.show()

## By terms

In [38]:
# Get distribution of metrics of company within the industry
query = f"""
    SELECT
        company_name,
        stock_code,
        fundamental->>'sector' AS sector,
        fundamental->>'industry' AS industry,
        fundamental->>'longBusinessSummary' AS summary
    FROM stock_info
    WHERE
        POSITION('machine learning' IN LOWER(fundamental->>'longBusinessSummary')) > 0
"""
df = pd.read_sql_query(query, engine).set_index('stock_code')
business_codes = df.index
df.head()

,company_name,sector,industry,summary
stock_code,,,,
IKE,Ikegps Group Ltd,Technology,Scientific & Technical Instruments,"ikeGPS Group Limited, together with its subsid..."
BRN,Brainchip Holdings Ltd,Technology,Semiconductors,BrainChip Holdings Ltd develops software and h...
CT1,Constellation Technologies Ltd,Technology,Information Technology Services,Constellation Technologies Limited engages in ...
DTI,DTI Group Ltd,Industrials,Security & Protection Services,"DTI Group Limited, together with its subsidiar..."
FGL,Frugl Group Ltd,Communication Services,Internet Content & Information,Frugl Group Limited engages in the development...


In [39]:
df.loc['BRN', 'summary']

'BrainChip Holdings Ltd develops software and hardware accelerated solutions for artificial intelligence and machine learning applications in North America, Oceania, Europe, the Middle East, and Asia. The company primarily focuses on development of Akida Neuromorphic Processor to provide ultra-low power and fast AI Edge Network for vision, audio, olfactory, and smart transducer applications. It also offers Akida IP, a neural processor; MetaTF, which is used for creation, training, and testing of neutral network; Akida enablement platforms, including Akida PCI, Edge AI box, and development kit and shuttle PC and raspberry Pi; AKD1000, a reference chip as a hardware accelerator; and 2nd generation akidaTM expands the benefits of event-based, neuromorphic processing for network models. In addition, the company provides MetaTF ML framework python packages, which includes Akida python package, an interface to the BrainChip Akida Neuromorphic System-on-Chip; CNN2SNN tool provides means to co

In [36]:
# Get distribution of metrics of company within the industry
query = f"""
    SELECT
        company_name,
        stock_code,
        fundamental->>'industry' AS industry,
        fundamental->>'marketCap' AS market_cap,
        fundamental->>'priceToBook' AS pb_ratio,
        fundamental->>'profitMargins' AS profit_margins,
        fundamental->>'trailingEps' AS earning_per_share,
        fundamental->>'enterpriseToEbitda' AS evebitda,
        fundamental->>'trailingPegRatio' AS peg_ratio,
        fundamental->>'debtToEquity' AS de_ratio
    FROM stock_info
    WHERE
        stock_code IN {tuple(business_codes)}

"""
df = pd.read_sql_query(query, engine)
df.head(5)

,company_name,stock_code,industry,market_cap,pb_ratio,profit_margins,earning_per_share,evebitda,peg_ratio,de_ratio
0,Brainchip Holdings Ltd,BRN,Semiconductors,653001280,39.374996,None,-0.02,-20.609,None,9.417
1,Constellation Technologies Ltd,CT1,Information Technology Services,2949460,2.0,-0.09039,None,-18.561,None,None
2,DTI Group Ltd,DTI,Security & Protection Services,5810961,1.4444445,-0.32254001,-0.01,-2.904,None,21.17
3,Frugl Group Ltd,FGL,Internet Content & Information,1522864,0.5833334,None,-0.1,-0.838,None,57.672
4,Icetana Ltd,ICE,Software—Application,5027932,4.7499995,-0.52728003,-0.01,-1.187,None,9.735


In [37]:
# Rerun the histogram, but add the company in it
metrics = []
# Only shows metric that has at least 90% of values
for metric in ['market_cap', 'pb_ratio', 'profit_margins', 'earning_per_share', 'evebitda', 'peg_ratio', 'de_ratio']:
    if len(df[~pd.isna(df[metric])]) / len(df) >= 0.9:
        metrics.append(metric)

fig = make_subplots(rows = len(metrics), cols = 1)
for i,metric in enumerate(metrics):
    fig.add_trace(
        go.Histogram(
            x = df[metric].apply(lambda x : float(x) if not pd.isna(x) else None), 
            marker = dict(color = 'blue'),
            showlegend = False
        ),
        row = i + 1, col = 1
    )
    fig.update_xaxes(title = metric, row = i + 1, col = 1)
fig.update_yaxes(showgrid = True, gridcolor = 'gray', title = 'Number of company')

fig.update_layout(
    width = 500,
    height = len(metrics) * 300,
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    font = dict(family = 'Lato', size = 18)
)
fig.show()

# Vietnam

In [10]:
engine = SQLModule.get_engine(country = 'vietnam')

In [11]:
# Collect from 'company_section.ipynb'
SECTOR = 'Financial Services'
INDUSTRY = 'Banks—Regional'

In [24]:
# Get distribution of metrics of company within the industry
query = f"""
    SELECT
        company_name,
        stock_code,
        fundamental->>'longBusinessSummary' AS summary
    FROM stock_info
    WHERE
        fundamental->>'sector' = '{SECTOR}'
        AND
        fundamental->>'industry' = '{INDUSTRY}'
"""
df = pd.read_sql_query(query, engine).set_index('stock_code')
business_codes = df.index.tolist()
df

,company_name,summary
stock_code,,
ACB,Asia Commercial Bank,Asia Commercial Joint Stock Bank provides vari...
EIB,Vietnam Export Import Commercial Joint Stock Bank,Vietnam Export Import Commercial Joint Stock B...
BID,Joint Stock Commercial Bank for Investment and...,Joint Stock Commercial Bank for Investment and...
CTG,Vietnam JSCmmercial Bank for Industry and Trade,Vietnam Joint Stock Commercial Bank for Indust...
HDB,Ho Chi Minh City Development Joint Stock Comme...,Ho Chi Minh City Development Joint Stock Comme...
MSB,Vietnam Maritime Commercial JS Bank,Vietnam Maritime Commercial Joint Stock Bank p...
LPB,Lien Viet Post Joint Stock Commercial Bank,Fortune Vietnam Joint Stock Commercial Bank pr...
MBB,Military Commercial Joint Stock Bank,Military Commercial Joint Stock Bank provides ...
OCB,Orient Commercial JS Bank,Orient Commercial Joint Stock Bank provides ba...


In [25]:
# Get distribution of metrics of company within the industry
query = f"""
    SELECT
        company_name,
        stock_code,
        fundamental,
        fundamental->>'industry' AS industry,
        fundamental->>'marketCap' AS market_cap,
        fundamental->>'priceToBook' AS pb_ratio,
        fundamental->>'profitMargins' AS profit_margins,
        fundamental->>'trailingEps' AS earning_per_share,
        fundamental->>'enterpriseToEbitda' AS evebitda,
        fundamental->>'trailingPegRatio' AS peg_ratio,
        fundamental->>'debtToEquity' AS de_ratio
    FROM stock_info
    WHERE
        stock_code IN {tuple(business_codes)}

"""
df = pd.read_sql_query(query, engine)
df.head(5)

,company_name,stock_code,fundamental,industry,market_cap,pb_ratio,profit_margins,earning_per_share,evebitda,peg_ratio,de_ratio
0,Asia Commercial Bank,ACB,"{'address1': '442 Nguyen Thi Minh Khai', 'addr...",Banks—Regional,None,1.4416018,0.51502997,3547.73,None,None,None
1,Joint Stock Commercial Bank for Investment and...,BID,"{'address1': 'BIDV Tower', 'address2': '194 Tr...",Banks—Regional,None,1.6714455,0.41626,3006.97,None,None,None
2,Vietnam JSCmmercial Bank for Industry and Trade,CTG,"{'address1': '108 Tran Hung Dao Street', 'addr...",Banks—Regional,None,1.4642454,0.43702,4018.64,None,None,None
3,Vietnam Export Import Commercial Joint Stock Bank,EIB,"{'address1': 'No. 29 - 29, Ly Thai To Street',...",Banks—Regional,None,1.4908612,0.45901,1441.05,None,None,None
4,Ho Chi Minh City Development Joint Stock Comme...,HDB,"{'address1': 'Ben Nghe Ward', 'address2': 'No ...",Banks—Regional,None,1.3422145,0.46227002,3747.52,None,None,None


In [29]:
df[df['stock_code'] == 'ACB']

,company_name,stock_code,fundamental,industry,market_cap,pb_ratio,profit_margins,earning_per_share,evebitda,peg_ratio,de_ratio
0,Asia Commercial Bank,ACB,"{'address1': '442 Nguyen Thi Minh Khai', 'addr...",Banks—Regional,None,1.4416018,0.51502997,3547.73,None,None,None


In [28]:
# Rerun the histogram, but add the company in it
metrics = []
# Only shows metric that has at least 90% of values
for metric in ['market_cap', 'pb_ratio', 'profit_margins', 'earning_per_share', 'evebitda', 'peg_ratio', 'de_ratio']:
    if len(df[~pd.isna(df[metric])]) / len(df) >= 0.9:
        metrics.append(metric)

fig = make_subplots(rows = len(metrics), cols = 1)
for i,metric in enumerate(metrics):
    fig.add_trace(
        go.Histogram(
            x = df[metric].apply(lambda x : float(x) if not pd.isna(x) else None), 
            marker = dict(color = 'blue'),
            showlegend = False
        ),
        row = i + 1, col = 1
    )
    fig.update_xaxes(title = metric, row = i + 1, col = 1)
fig.update_yaxes(showgrid = True, gridcolor = 'gray', title = 'Number of company')

fig.update_layout(
    width = 500,
    height = len(metrics) * 300,
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    font = dict(family = 'Lato', size = 18)
)
fig.show()

# United States

In [105]:
engine = SQLModule.get_engine(country = 'united_states')

## By industry

In [106]:
# Collect from 'company_section.ipynb'
SECTOR = 'Technology'
INDUSTRY = 'Information Technology Services'

In [112]:
# Get distribution of metrics of company within the industry
query = f"""
    SELECT
        company_name,
        stock_code,
        fundamental->>'longBusinessSummary' AS summary,
        fundamental->>'debtToEquity' AS de_ratio
    FROM stock_info
    WHERE
        fundamental->>'sector' = '{SECTOR}'
        AND
        fundamental->>'industry' = '{INDUSTRY}'
"""
df = pd.read_sql_query(query, engine).set_index('stock_code')
df

,company_name,summary,de_ratio
stock_code,,,
ACN,Accenture plc,Accenture plc provides strategy and consulting...,14.127
ALYAF,Alithya Group Inc,Alithya Group Inc. provides strategy and digit...,None
ALYI,Alternet Systems Inc,"Alternet Systems, Inc., through its subsidiari...",None
AMADF,Amadeus IT Group S.A,"Amadeus IT Group, S.A., together with its subs...",77.819
AMADY,Amadeus IT Holding SA PK,"Amadeus IT Group, S.A., together with its subs...",77.819
...,...,...,...
WIZEY,Wise plc,Wise plc provides cross-border and domestic fi...,4.465
WNS,WNS Holdings Ltd,"WNS (Holdings) Limited, a business process man...",60.567
WYLDF,Wyld Networks AB (publ),"Wyld Networks AB (publ), together with its sub...",None


In [113]:
focused_df = df[df['summary'].apply(lambda x : 'AI' in x or 'Artificial Intelligence' in x)]
# focused_df = df[~pd.isna(df['de_ratio'])]
# focused_df = df
business_codes = focused_df.index.tolist()

In [114]:
# Get distribution of metrics of company within the industry
query = f"""
    SELECT
        company_name,
        stock_code,
        fundamental,
        fundamental->>'industry' AS industry,
        CAST(fundamental->>'marketCap' AS FLOAT) AS market_cap,
        CAST(fundamental->>'priceToBook' AS FLOAT) AS pb_ratio,
        CAST(fundamental->>'profitMargins' AS FLOAT) AS profit_margins,
        CAST(fundamental->>'trailingEps' AS FLOAT) AS earning_per_share,
        CAST(fundamental->>'enterpriseToEbitda' AS FLOAT) AS evebitda,
        CAST(fundamental->>'trailingPegRatio' AS FLOAT) AS peg_ratio,
        CAST(fundamental->>'debtToEquity' AS FLOAT) AS de_ratio
    FROM stock_info
    WHERE
        stock_code IN {tuple(business_codes)}

"""
df = pd.read_sql_query(query, engine)
df.head(5)

,company_name,stock_code,fundamental,industry,market_cap,pb_ratio,profit_margins,earning_per_share,evebitda,peg_ratio,de_ratio
0,Accenture plc,ACN,"{'address1': '1 Grand Canal Square', 'address2...",Information Technology Services,2.261923e+11,7.993237,0.11194,11.93,19.628,2.7873,14.127
1,Alithya Group Inc,ALYAF,"{'address1': '1100, Robert-Bourassa Boulevard'...",Information Technology Services,1.143577e+08,0.626362,NaN,-0.02,NaN,NaN,NaN
2,Applied Digital Corporation,APLD,"{'address1': '3811 Turtle Creek Boulevard', 'a...",Information Technology Services,1.852627e+09,5.709636,-0.74811,-1.23,142.545,NaN,143.294
3,Appen Limited,APPEF,"{'address1': '9 Help Street', 'address2': 'Lev...",Information Technology Services,3.765013e+08,4.044818,-0.37319,-0.61,-49.249,NaN,14.153
4,Appen Limited,APXYY,"{'address1': '9 Help Street', 'address2': 'Lev...",Information Technology Services,3.559671e+08,1.858543,-0.37319,-0.30,-20.541,NaN,14.153


In [118]:
df.sort_values('evebitda', ascending = False)

,company_name,stock_code,fundamental,industry,market_cap,pb_ratio,profit_margins,earning_per_share,evebitda,peg_ratio,de_ratio
2,Applied Digital Corporation,APLD,"{'address1': '3811 Turtle Creek Boulevard', 'a...",Information Technology Services,1.852627e+09,5.709636,-0.74811,-1.23,142.545,NaN,143.294
11,Grid Dynamics Holdings Inc,GDYN,"{'address1': '5000 Executive Parkway', 'addres...",Information Technology Services,1.712818e+09,3.936721,0.00737,0.03,113.299,NaN,3.205
18,Innodata Inc,INOD,"{'address1': '55 Challenger Road', 'address2':...",Information Technology Services,1.249623e+09,26.194529,0.14568,0.61,68.908,NaN,10.385
12,Globant SA,GLOB,"{'address1': '5 rue Guillaume Kroll', 'city': ...",Information Technology Services,9.468119e+09,5.466819,0.07391,3.84,29.201,1.8736,13.235
16,Information Services Group Inc,III,"{'address1': '2187 Atlantic Street', 'city': '...",Information Technology Services,1.643922e+08,1.700508,-0.01201,-0.06,26.638,0.8251,72.621
17,Infosys Ltd ADR,INFY,"{'address1': 'Plot No. 44/97 A', 'address2': '...",Information Technology Services,9.431403e+10,8.740883,0.17159,0.77,21.152,3.0440,9.697
0,Accenture plc,ACN,"{'address1': '1 Grand Canal Square', 'address2...",Information Technology Services,2.261923e+11,7.993237,0.11194,11.93,19.628,2.7873,14.127
20,Parsons Corp,PSN,"{'address1': '14291 Park Meadow Drive', 'addre...",Information Technology Services,9.921332e+09,4.290897,0.01208,0.70,18.987,NaN,58.304
15,International Business Machines,IBM,"{'address1': 'One New Orchard Road', 'city': '...",Information Technology Services,2.074996e+11,8.487519,0.10222,6.87,17.241,7.4313,245.112
31,TravelSky Technology Ltd ADR,TSYHY,"{'address1': 'China Resources Building', 'addr...",Information Technology Services,3.963095e+09,1.853366,0.20287,0.74,14.810,NaN,6.638


In [119]:
target_company = df[df['stock_code'] == 'GMM'].iloc[0]
# target_company = None
# Rerun the histogram, but add the company in it
fig = make_subplots(rows = 4, cols = 2)
metrics = ['market_cap', 'pb_ratio', 'profit_margins', 'earning_per_share', 'evebitda', 'peg_ratio', 'de_ratio']

for i,metric in enumerate(metrics):
    fig.add_trace(
        go.Histogram(
            x = df[metric].apply(lambda x : float(x) if not pd.isna(x) else None), 
            marker = dict(color = 'blue'),
            showlegend = False
        ),
        row = (i // 2) + 1, col = (i % 2) + 1
    )
    if target_company is not None:
        fig.add_trace(
            go.Scatter(
                x = [float(target_company[metric]) if target_company[metric] is not None else None],
                y = [0],
                marker = dict(color = 'red'),
                showlegend = False
            ),
            row = (i // 2) + 1, col = (i % 2) + 1
        )
    fig.update_xaxes(title = metric, row = (i // 2) + 1, col = (i % 2) + 1)
fig.update_yaxes(showgrid = True, gridcolor = 'gray', title = 'Number of company')

fig.update_layout(
    width = 1000,
    height = 1000,
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    font = dict(family = 'Lato', size = 18)
)
fig.show()

In [ ]:
# The same company